# Chatbot de Autos con RAG (LangChain + OpenAI + FAISS)

Chatbot que permite hacer búsquedas semánticas sobre un dataset de autos usando:
- RAG (Retrieval Augmented Generation)
- FAISS - Vector store rápido y local
- OpenAI Embeddings - Para vectorizar los datos
- LangChain - Orquestación simple


!pip install langchain langchain-openai langchain-community langchain-core faiss-cpu pandas --quiet

import os
import pandas as pd
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [1]:
!pip install langchain langchain-openai langchain-community langchain-core faiss-cpu pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os
import pandas as pd
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate


In [3]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 3. Configurar API Key

Obtené tu API key en: https://platform.openai.com/api-keys

In [ ]:
# Reemplaza con tu API key
os.environ["OPENAI_API_KEY"] = ""

print("✅ API Key configurada")

✅ API Key configurada


## 4. Cargar dataset

In [6]:
df = pd.read_csv("/content/drive/MyDrive/deep learning/tarea autos/Notebook Interactivo/cars 2.csv")
print(f"📊 Dataset cargado: {df.shape[0]} autos, {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")
df.head()

📊 Dataset cargado: 38531 autos, 30 columnas

Columnas: ['manufacturer_name', 'model_name', 'transmission', 'color', 'odometer_value', 'year_produced', 'engine_fuel', 'engine_has_gas', 'engine_type', 'engine_capacity', 'body_type', 'has_warranty', 'state', 'drivetrain', 'price_usd', 'is_exchangeable', 'location_region', 'number_of_photos', 'up_counter', 'feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'duration_listed']


,manufacturer_name,model_name,transmission,color,odometer_value,year_produced,engine_fuel,engine_has_gas,engine_type,engine_capacity,...,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,duration_listed
0,Subaru,Outback,automatic,silver,190000,2010,gasoline,False,gasoline,2.5,...,True,True,True,False,True,False,True,True,True,16
1,Subaru,Outback,automatic,blue,290000,2002,gasoline,False,gasoline,3.0,...,True,False,False,True,True,False,False,False,True,83
2,Subaru,Forester,automatic,red,402000,2001,gasoline,False,gasoline,2.5,...,True,False,False,False,False,False,False,True,True,151
3,Subaru,Impreza,mechanical,blue,10000,1999,gasoline,False,gasoline,3.0,...,False,False,False,False,False,False,False,False,False,86
4,Subaru,Legacy,automatic,black,280000,2001,gasoline,False,gasoline,2.5,...,True,False,True,True,False,False,False,False,True,7


## 5. Convertir datos a formato narrativo

Cada auto se convierte en una descripción en texto natural para mejorar la búsqueda semántica.

In [7]:
def row_to_text(row):
    """
    Convierte una fila del DataFrame a texto descriptivo narrativo.
    """
    # Manejar valores nulos
    def safe_val(val, default="desconocido"):
        return val if pd.notna(val) else default

    text = f"""{safe_val(row['manufacturer_name'])} {safe_val(row['model_name'])} del año {safe_val(row['year_produced'], 'N/A')}.
Precio: ${safe_val(row['price_usd'], 0):,.0f} USD.
Motor: {safe_val(row['engine_fuel'])} de {safe_val(row['engine_capacity'], 'N/A')} litros.
Transmisión: {safe_val(row['transmission'])}.
Kilometraje: {safe_val(row['odometer_value'], 0):,.0f} km.
Tipo de carrocería: {safe_val(row['body_type'])}.
Color: {safe_val(row['color'])}.
Tracción: {safe_val(row['drivetrain'])}.
Estado: {safe_val(row['state'])}.
"""
    return text.strip()

# Probar con el primer auto
print("Ejemplo de descripción narrativa:\n")
print(row_to_text(df.iloc[0]))
print("\n✅ Función de conversión creada")

Ejemplo de descripción narrativa:

Subaru Outback del año 2010.
Precio: $10,900 USD.
Motor: gasoline de 2.5 litros.
Transmisión: automatic.
Kilometraje: 190,000 km.
Tipo de carrocería: universal.
Color: silver.
Tracción: all.
Estado: owned.

✅ Función de conversión creada


## 6. Crear documentos para vectorización

Convertimos cada auto a un `Document` de LangChain con texto + metadata.

In [8]:
documents = []

for idx, row in df.iterrows():
    # Crear texto narrativo
    text = row_to_text(row)

    # Crear metadata con info estructurada
    metadata = {
        "manufacturer": str(row['manufacturer_name']),
        "model": str(row['model_name']),
        "year": str(row['year_produced']),
        "price": float(row['price_usd']) if pd.notna(row['price_usd']) else 0,
        "color": str(row['color']),
        "body_type": str(row['body_type']),
        "engine_fuel": str(row['engine_fuel']),
        "index": idx
    }

    # Crear documento
    doc = Document(page_content=text, metadata=metadata)
    documents.append(doc)

print(f"✅ {len(documents)} documentos creados")
print(f"\nEjemplo del primer documento:")
print(f"Texto: {documents[0].page_content[:200]}...")
print(f"Metadata: {documents[0].metadata}")

✅ 38531 documentos creados

Ejemplo del primer documento:
Texto: Subaru Outback del año 2010.
Precio: $10,900 USD.
Motor: gasoline de 2.5 litros.
Transmisión: automatic.
Kilometraje: 190,000 km.
Tipo de carrocería: universal.
Color: silver.
Tracción: all.
Estado: o...
Metadata: {'manufacturer': 'Subaru', 'model': 'Outback', 'year': '2010', 'price': 10900.0, 'color': 'silver', 'body_type': 'universal', 'engine_fuel': 'gasoline', 'index': 0}


## 7. Crear embeddings y vector store con FAISS

**Nota:** Este paso tarda un poco la primera vez (genera embeddings para todos los autos).
Después se guarda en disco y se carga instantáneamente.

In [9]:
import os.path

VECTOR_STORE_PATH = "./faiss_autos_index"

# Inicializar embeddings de OpenAI
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"  # Modelo más económico y rápido
)

# Verificar si ya existe el vector store
if os.path.exists(VECTOR_STORE_PATH):
    print("📂 Cargando vector store existente desde disco...")
    vectorstore = FAISS.load_local(
        VECTOR_STORE_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("✅ Vector store cargado desde disco")
else:
    print("🔄 Creando vector store por primera vez (esto puede tardar 1-2 minutos)...")
    vectorstore = FAISS.from_documents(documents, embeddings)

    # Guardar en disco para uso futuro
    vectorstore.save_local(VECTOR_STORE_PATH)
    print(f"✅ Vector store creado y guardado en '{VECTOR_STORE_PATH}'")

print(f"\n📊 Total de vectores en el store: {vectorstore.index.ntotal}")

🔄 Creando vector store por primera vez (esto puede tardar 1-2 minutos)...
✅ Vector store creado y guardado en './faiss_autos_index'

📊 Total de vectores en el store: 38531


## 8. Configurar el LLM y crear el chain de RetrievalQA

In [10]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Inicializar modelo de OpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,  # Respuestas más determinísticas
)

# Crear retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Template personalizado en español
template = """Sos un asistente experto en autos que ayuda a los usuarios a encontrar información sobre vehículos.

Usa el siguiente contexto para responder la pregunta del usuario. El contexto contiene información sobre autos reales del dataset.

Contexto:
{context}

Pregunta: {question}

Instrucciones:
- Respondé en español de forma clara y concisa
- Si encontrás varios autos relevantes, mencioná los más destacados (máximo 5)
- Incluí detalles específicos como marca, modelo, año, precio cuando sea relevante
- Si no hay información suficiente en el contexto, decí que no encontraste autos que coincidan
- Podés hacer comparaciones si el usuario lo pide

Respuesta:"""

prompt = ChatPromptTemplate.from_template(template)

# Función para formatear documentos
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Crear el chain RAG usando LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Chain RAG creado")

✅ Chain RAG creado


## 9. Función para hacer preguntas

In [11]:
# Inicializar historial de conversación
historial_conversacion = []

def preguntar(pregunta: str, mostrar_fuentes: bool = False):
    """
    Hace una pregunta al chatbot usando RAG con memoria de conversación.

    Args:
        pregunta: La pregunta en lenguaje natural
        mostrar_fuentes: Si True, muestra los autos usados como fuente

    Returns:
        La respuesta del chatbot
    """
    try:
        # 1. Obtener documentos relevantes
        docs = retriever.invoke(pregunta)

        # 2. Mostrar fuentes si se solicita
        if mostrar_fuentes:
            print("\n" + "="*60)
            print("📚 FUENTES USADAS (Top 5 autos más relevantes):")
            print("="*60)
            for i, doc in enumerate(docs, 1):
                print(f"\n{i}. {doc.metadata['manufacturer']} {doc.metadata['model']} ({doc.metadata['year']})")
                print(f"   Precio: ${doc.metadata['price']:,.0f} USD")
                print(f"   Color: {doc.metadata['color']} | Tipo: {doc.metadata['body_type']}")
            print("="*60 + "\n")

        # 3. Formatear el contexto
        context = "\n\n".join([doc.page_content for doc in docs])

        # 4. Formatear el historial de conversación
        historial_texto = ""
        if historial_conversacion:
            historial_texto = "\n\nHistorial de conversación previa:\n"
            for msg in historial_conversacion[-6:]:  # Solo las últimas 3 interacciones (6 mensajes)
                historial_texto += f"{msg['role']}: {msg['content']}\n"

        # 5. Crear el prompt con historial
        prompt_text = f"""Sos un asistente experto en autos que ayuda a los usuarios a encontrar información sobre vehículos.

Usa el siguiente contexto para responder la pregunta del usuario. El contexto contiene información sobre autos reales del dataset.

Contexto:
{context}
{historial_texto}

Pregunta actual: {pregunta}

Instrucciones:
- Respondé en español de forma clara y concisa
- Si el usuario hace referencia a preguntas anteriores, usa el historial para mantener contexto
- Si encontrás varios autos relevantes, mencioná los más destacados (máximo 5)
- Incluí detalles específicos como marca, modelo, año, precio cuando sea relevante
- Si no hay información suficiente en el contexto, decí que no encontraste autos que coincidan
- Podés hacer comparaciones si el usuario lo pide

Respuesta:"""

        # 6. Invocar el LLM directamente
        response = llm.invoke(prompt_text)

        # 7. Guardar la interacción en el historial
        historial_conversacion.append({
            "role": "Usuario",
            "content": pregunta
        })
        historial_conversacion.append({
            "role": "Asistente",
            "content": response.content
        })

        # 8. Extraer el contenido de la respuesta
        return response.content

    except Exception as e:
        import traceback
        error_detail = traceback.format_exc()
        return f"❌ Error: {str(e)}\n\nDetalles:\n{error_detail}"

def limpiar_historial():
    """Limpia el historial de conversación"""
    global historial_conversacion
    historial_conversacion = []
    print("🧹 Historial limpiado")

print("✅ Función de consulta creada con memoria (sin LCEL)")

✅ Función de consulta creada con memoria (sin LCEL)


## 10. Ejemplos de búsquedas semánticas

Probá diferentes tipos de preguntas:

In [12]:
# Limpiar historial para empezar de cero
limpiar_historial()

# Primera pregunta
print("=" * 60)
print("🧑 Pregunta 1: Muéstrame SUVs Toyota")
print("=" * 60)
respuesta1 = preguntar("Muéstrame SUVs Toyota")
print(f"\n🤖 Respuesta:\n{respuesta1}")

# Segunda pregunta que hace referencia a la anterior
print("\n\n" + "=" * 60)
print("🧑 Pregunta 2: ¿Cuál es el más barato de esos?")
print("=" * 60)
print("(Nota: Esta pregunta hace referencia a los autos de la pregunta anterior)")
respuesta2 = preguntar("¿Cuál es el más barato de esos?")
print(f"\n🤖 Respuesta:\n{respuesta2}")

# Tercera pregunta para seguir el contexto
print("\n\n" + "=" * 60)
print("🧑 Pregunta 3: ¿Y tiene garantía?")
print("=" * 60)
print("(Nota: Esta pregunta hace referencia al auto más barato mencionado)")
respuesta3 = preguntar("¿Y tiene garantía?")
print(f"\n🤖 Respuesta:\n{respuesta3}")

print("\n\n" + "=" * 60)
print("✅ Como ves, el chatbot recuerda el contexto de la conversación!")
print("=" * 60)

🧹 Historial limpiado
🧑 Pregunta 1: Muéstrame SUVs Toyota

🤖 Respuesta:
Aquí tienes una lista de SUVs Toyota RAV4 disponibles según el contexto:

1. **Toyota RAV4 2014**
   - Precio: $18,500 USD
   - Motor: diesel de 2.2 litros
   - Transmisión: automática
   - Kilometraje: 157,000 km
   - Color: marrón

2. **Toyota RAV4 2005**
   - Precio: $7,700 USD
   - Motor: diesel de 2.0 litros
   - Transmisión: mecánica
   - Kilometraje: 237,000 km
   - Color: otro

3. **Toyota RAV4 2007**
   - Precio: $10,500 USD
   - Motor: diesel de 2.2 litros
   - Transmisión: mecánica
   - Kilometraje: 192,550 km
   - Color: gris

4. **Toyota RAV4 2015**
   - Precio: $20,100 USD
   - Motor: diesel de 2.2 litros
   - Transmisión: automática
   - Kilometraje: 180,000 km
   - Color: otro

5. **Toyota RAV4 2002**
   - Precio: $5,950 USD
   - Motor: diesel de 2.0 litros
   - Transmisión: mecánica
   - Kilometraje: 400,000 km
   - Color: plateado

Si necesitas más información sobre alguno de estos modelos o compar

In [13]:
# Ejemplo 1: Búsqueda por características
print("\n" + "="*60)
print("❓ Pregunta: Muéstrame autos SUV que sean económicos")
print("="*60)
respuesta = preguntar("Muéstrame autos SUV que sean económicos", mostrar_fuentes=True)
print(f"\n🤖 Respuesta:\n{respuesta}")


❓ Pregunta: Muéstrame autos SUV que sean económicos

📚 FUENTES USADAS (Top 5 autos más relevantes):

1. Hyundai Santa Fe (2003)
   Precio: $6,500 USD
   Color: silver | Tipo: suv

2. Renault Duster (2017)
   Precio: $5,900 USD
   Color: brown | Tipo: suv

3. Hyundai Santa Fe (2003)
   Precio: $5,500 USD
   Color: silver | Tipo: suv

4. Suzuki SX4 S-Cross (2014)
   Precio: $5,500 USD
   Color: brown | Tipo: suv

5. Hyundai Santa Fe (2003)
   Precio: $5,300 USD
   Color: silver | Tipo: suv


🤖 Respuesta:
Aquí tienes una lista de SUVs que son económicos según el contexto proporcionado:

1. **Renault Duster 2017**
   - **Precio**: $5,900 USD
   - **Motor**: gasolina de 2.0 litros
   - **Transmisión**: automática
   - **Kilometraje**: 30,000 km
   - **Color**: marrón
   - **Estado**: emergencia

2. **Hyundai Santa Fe 2003**
   - **Precio**: $5,300 USD
   - **Motor**: diesel de 2.0 litros
   - **Transmisión**: mecánica
   - **Kilometraje**: 380,000 km
   - **Color**: plateado
   - **Estado*

In [14]:
# Ejemplo 4: Búsqueda por precio
print("\n" + "="*60)
print("❓ Pregunta: Necesito un auto familiar por menos de $10,000")
print("="*60)
respuesta = preguntar("Necesito un auto familiar por menos de $10,000", mostrar_fuentes=True)
print(f"\n🤖 Respuesta:\n{respuesta}")


❓ Pregunta: Necesito un auto familiar por menos de $10,000

📚 FUENTES USADAS (Top 5 autos más relevantes):

1. Ford Focus (2000)
   Precio: $4,000 USD
   Color: silver | Tipo: universal

2. Nissan Micra (2007)
   Precio: $1,500 USD
   Color: red | Tipo: hatchback

3. Dodge Neon (2000)
   Precio: $2,300 USD
   Color: silver | Tipo: sedan

4. Hyundai Sonata (1994)
   Precio: $500 USD
   Color: black | Tipo: sedan

5. Nissan Almera (2000)
   Precio: $650 USD
   Color: silver | Tipo: hatchback


🤖 Respuesta:
Aquí tienes una lista de autos familiares que están por debajo de $10,000 según el contexto proporcionado:

1. **Ford Focus 2000**
   - **Precio**: $4,000 USD
   - **Motor**: gasolina de 1.6 litros
   - **Transmisión**: mecánica
   - **Kilometraje**: 270,000 km
   - **Tipo de carrocería**: universal
   - **Color**: plateado
   - **Estado**: emergencia

2. **Nissan Micra 2007**
   - **Precio**: $1,500 USD
   - **Motor**: gasolina de 1.3 litros
   - **Transmisión**: mecánica
   - **Kilo

In [15]:
# Ejemplo 5: Comparación
print("\n" + "="*60)
print("❓ Pregunta: Compará autos Honda vs Mazda")
print("="*60)
respuesta = preguntar("Compará autos Honda vs Mazda", mostrar_fuentes=True)
print(f"\n🤖 Respuesta:\n{respuesta}")


❓ Pregunta: Compará autos Honda vs Mazda

📚 FUENTES USADAS (Top 5 autos más relevantes):

1. Honda CR-Z (2014)
   Precio: $11,000 USD
   Color: grey | Tipo: hatchback

2. Honda CRX (1987)
   Precio: $1,500 USD
   Color: black | Tipo: coupe

3. Honda CR-Z (2010)
   Precio: $9,500 USD
   Color: red | Tipo: coupe

4. Honda CR-Z (2010)
   Precio: $8,200 USD
   Color: black | Tipo: hatchback

5. Honda CR-Z (2010)
   Precio: $8,200 USD
   Color: black | Tipo: coupe


🤖 Respuesta:
En el contexto proporcionado, solo tengo información sobre autos Honda. No hay datos sobre vehículos Mazda. Aquí tienes un resumen de los autos Honda disponibles:

1. **Honda CR-Z 2014**
   - **Precio**: $11,000 USD
   - **Motor**: hybrid-petrol de 1.5 litros
   - **Transmisión**: mecánica
   - **Kilometraje**: 91,893 km
   - **Tipo de carrocería**: hatchback
   - **Color**: gris
   - **Estado**: owned

2. **Honda CR-Z 2010 (Automático)**
   - **Precio**: $9,500 USD
   - **Motor**: hybrid-petrol de 1.5 litros
   - 

## 11. Chatbot interactivo

Escribí tus preguntas en lenguaje natural.

**Comandos:**
- `salir` - Terminar el chat
- `fuentes` - Activar/desactivar mostrar fuentes

In [16]:
print("=" * 60)
print("🚗 Chatbot de Autos con RAG + Memoria")
print("Preguntame sobre autos en lenguaje natural.")
print("")
print("Comandos: 'salir', 'fuentes' (activar/desactivar), 'limpiar' (borrar historial)")
print("=" * 60)

mostrar_fuentes = False
print(f"\n💡 Fuentes: {'Activadas' if mostrar_fuentes else 'Desactivadas'} (escribe 'fuentes' para cambiar)")
print(f"📚 Historial: Activo (escribe 'limpiar' para reiniciar la conversación)\n")

while True:
    pregunta_usuario = input("\n🧑 Vos: ")

    if pregunta_usuario.strip().lower() == "salir":
        print("\n¡Chau! 👋")
        break

    elif pregunta_usuario.strip().lower() == "fuentes":
        mostrar_fuentes = not mostrar_fuentes
        print(f"\n💡 Fuentes: {'Activadas ✅' if mostrar_fuentes else 'Desactivadas ❌'}")
        continue

    elif pregunta_usuario.strip().lower() == "limpiar":
        limpiar_historial()
        continue

    if pregunta_usuario.strip():
        try:
            respuesta = preguntar(pregunta_usuario, mostrar_fuentes=mostrar_fuentes)
            print(f"\n🤖 Bot: {respuesta}")
        except Exception as e:
            print(f"\n❌ Error: {e}")

🚗 Chatbot de Autos con RAG + Memoria
Preguntame sobre autos en lenguaje natural.

Comandos: 'salir', 'fuentes' (activar/desactivar), 'limpiar' (borrar historial)

💡 Fuentes: Desactivadas (escribe 'fuentes' para cambiar)
📚 Historial: Activo (escribe 'limpiar' para reiniciar la conversación)


🧑 Vos: Muéstrame SUVs Toyota

🤖 Bot: Aquí tienes una lista de SUVs Toyota disponibles según el contexto proporcionado:

1. **Toyota RAV4 2014**
   - **Precio**: $18,500 USD
   - **Motor**: diesel de 2.2 litros
   - **Transmisión**: automática
   - **Kilometraje**: 157,000 km
   - **Color**: marrón
   - **Tracción**: all
   - **Estado**: owned

2. **Toyota RAV4 2005**
   - **Precio**: $7,700 USD
   - **Motor**: diesel de 2.0 litros
   - **Transmisión**: mecánica
   - **Kilometraje**: 237,000 km
   - **Color**: otro
   - **Tracción**: all
   - **Estado**: owned

3. **Toyota RAV4 2007**
   - **Precio**: $10,500 USD
   - **Motor**: diesel de 2.2 litros
   - **Transmisión**: mecánica
   - **Kilometraje*

## 12. Búsqueda directa en el vector store (sin LLM)

También podés hacer búsquedas directas para ver los autos más similares sin generar respuesta con el LLM.

In [17]:
def busqueda_directa(query: str, k: int = 5):
    """
    Búsqueda directa por similitud sin usar el LLM.
    """
    docs = vectorstore.similarity_search(query, k=k)

    print(f"\n🔍 Top {k} autos más similares a: '{query}'\n")
    print("="*60)

    for i, doc in enumerate(docs, 1):
        print(f"\n{i}. {doc.metadata['manufacturer']} {doc.metadata['model']} ({doc.metadata['year']})")
        print(f"   💰 Precio: ${doc.metadata['price']:,.0f} USD")
        print(f"   🎨 Color: {doc.metadata['color']}")
        print(f"   🚗 Tipo: {doc.metadata['body_type']}")
        print(f"   ⛽ Combustible: {doc.metadata['engine_fuel']}")

    print("\n" + "="*60)

# Ejemplo
busqueda_directa("sedán azul automático económico", k=5)


🔍 Top 5 autos más similares a: 'sedán azul automático económico'


1. Suzuki SX4 (2007)
   💰 Precio: $5,950 USD
   🎨 Color: blue
   🚗 Tipo: hatchback
   ⛽ Combustible: gasoline

2. Seat Cordoba (2002)
   💰 Precio: $3,200 USD
   🎨 Color: blue
   🚗 Tipo: sedan
   ⛽ Combustible: gasoline

3. Seat Cordoba (2000)
   💰 Precio: $2,050 USD
   🎨 Color: blue
   🚗 Tipo: sedan
   ⛽ Combustible: gasoline

4. Seat Cordoba (1997)
   💰 Precio: $1,650 USD
   🎨 Color: blue
   🚗 Tipo: sedan
   ⛽ Combustible: gas

5. Seat Cordoba (2001)
   💰 Precio: $2,200 USD
   🎨 Color: blue
   🚗 Tipo: sedan
   ⛽ Combustible: gasoline

